# 04: Açıklanabilir Yapay Zeka (XAI) (Kişi 4)

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Bu notebook'ta yapılanlar:**
1. Açıklanabilir yapay zeka yöntemlerinin araştırılması
2. Projeye uygun yöntemlerin seçilmesi (SHAP ve LIME) ve gerekçesi
3. Ekibin ortak kullanacağı XAI fonksiyonunun yazılması (`src/explain.py`)
4. Fonksiyonun ekip tarafından nasıl kullanılacağı ve sonuçların nasıl yorumlanacağı

## 1. Kurulum

In [ ]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

## 2. Açıklanabilir Yapay Zeka (XAI) nedir, neden gerekli?

Random Forest, XGBoost veya SVM gibi modeller yüksek doğruluk verse de **nasıl karar verdiklerini doğrudan göstermezler** (kara kutu modeller). Sağlık gibi alanlarda bir modelin "bu hastada inme riski yüksek" demesi yetmez; doktorun **neden** bu sonuca varıldığını görebilmesi gerekir. XAI yöntemleri, modelin kararlarını insanların anlayabileceği şekilde açıklar.

XAI yöntemleri iki ana gruba ayrılır:

| | Global açıklama | Lokal açıklama |
|---|---|---|
| **Soru** | Model genel olarak hangi özelliklere dayanıyor? | Model **bu hasta** için neden bu kararı verdi? |
| **Örnek** | "Yaş ve büyük tansiyon en önemli iki özellik" | "Bu hastada risk yüksek çünkü yaşı 65 ve tansiyonu 180" |

### İncelenen yöntemler

| Yöntem | Türü | Kısa açıklama |
|---|---|---|
| **Feature Importance** (ağaç modelleri) | Global | Ağaçlardaki bölünmelere göre önem. Hızlı ama yönü (riski artırıyor mu azaltıyor mu) göstermez, yanıltıcı olabilir |
| **Permutation Importance** | Global | Bir özelliği karıştırınca performans ne kadar düşüyor? Modelden bağımsız ama yön bilgisi yok |
| **SHAP** (SHapley Additive exPlanations) | Global + lokal | Oyun teorisindeki Shapley değerlerine dayanır. Her özelliğin tahmine katkısını adil şekilde dağıtır, yönü de gösterir |
| **LIME** (Local Interpretable Model-agnostic Explanations) | Lokal | Tek bir örneğin etrafında yapay veriler üretip basit bir doğrusal model kurar ve kararı onunla açıklar |

### Seçilen yöntemler: SHAP + LIME

- **SHAP** global açıklama için seçildi: matematiksel temeli güçlüdür, hem önem sırasını hem de etkinin yönünü gösterir. Ağaç modelleri için özel hızlı bir sürümü (TreeExplainer) vardır.
- **LIME** lokal açıklama için seçildi: modelden bağımsızdır (her modelde aynı şekilde çalışır) ve tek bir hasta için anlaşılır bir açıklama verir.
- İki yöntemi birlikte kullanmak, sonuçların birbirini doğrulayıp doğrulamadığını görmemizi sağlar.

### SHAP grafikleri nasıl okunur? (ekip için rehber)

- **Bar grafiği:** Çubuk ne kadar uzunsa özellik o kadar önemli.
- **Özet (beeswarm) grafiği:** Her nokta bir hasta. **Kırmızı** = özelliğin değeri yüksek, **mavi** = düşük. Nokta **sağdaysa** inme riskini **artırıyor**, soldaysa azaltıyor. Örnek: AGE satırında kırmızı noktalar sağdaysa "yaş arttıkça risk artıyor" denir.
- **LIME grafiği:** Yeşil çubuklar "inme var" kararını destekleyen, kırmızı çubuklar karşı çıkan koşullardır (örn. `AGE > 60`).

## 3. Ortak XAI fonksiyonu

Fonksiyonlar `src/explain.py` dosyasındadır. **Ekipteki herkes** bunu kendi notebook'una import edip tek satırla kullanır (Colab'da `config.py`, `preprocessing.py`, `metrics.py` ve `explain.py` dosyalarını sol menüden yükleyin):

```python
from explain import explain
result = explain(model, X_train, X_test, model_name="SVM")
```

**Ekibe notlar:**
- Model `predict_proba` desteklemiyorsa (ör. `SVC()`), değerlendirme fonksiyonlarıyla aynı skor kullanılır: `decision_function`'ın sigmoid'i (`metrics.predict_score`).
- Model bir pipeline olabilir (ör. Kişi 2'nin `build_pipeline()` çıktısı). SMOTE adımı otomatik atlanır; feature engineering ve one-hot ile eklenen yeni sütunlar (`SEX_female`, `bp_stage_stage2` vb.) SHAP grafiklerinde kendi isimleriyle görünür.
- Fonksiyon model türünü kendisi algılar: ağaç modellerinde TreeExplainer, lineer modellerde LinearExplainer, KNN/SVM'de KernelExplainer kullanır. KernelExplainer yavaş olduğu için test verisinden 150 satırlık örneklem alınır.
- Lineer ve XGBoost modellerinde SHAP değerleri log-odds, diğerlerinde olasılık ölçeğindedir. Modelleri karşılaştırırken ham sayılar yerine `compare_importance()` fonksiyonunun verdiği yüzdeler kullanılmalıdır.

In [ ]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

from explain import compare_importance, explain, lime_local, shap_global

## 4. Fonksiyonun içindeki adımlar

| Fonksiyon | Ne yapar | Ne döndürür |
|---|---|---|
| `shap_global(model, X_train, X_test)` | Model türünü algılar, uygun SHAP açıklayıcısını seçer, bar ve özet grafiklerini çizer | Özelliklerin ortalama mutlak SHAP değerine göre önem tablosu |
| `lime_local(model, X_train, X_test, index)` | Seçilen hasta için LIME açıklaması üretir. İndeks verilmezse modelin en riskli gördüğü hastayı seçer | (koşul, etki) listesi, ör. `("AGE > 60", +0.12)` |
| `explain(model, X_train, X_test, model_name)` | İkisini birlikte çalıştırır | `{"shap_importance": ..., "lime": ...}` |
| `compare_importance(results)` | Birden fazla modelin SHAP önemlerini yüzdeye çevirip yan yana koyar | Karşılaştırma tablosu |

**Model türüne göre seçilen SHAP açıklayıcısı:**

| Model | Açıklayıcı | Hız |
|---|---|---|
| Random Forest, XGBoost, Decision Tree | `TreeExplainer` | Çok hızlı, tüm test verisi kullanılır |
| Logistic Regression | `LinearExplainer` | Çok hızlı |
| KNN, SVM | `KernelExplainer` | Yavaş, test verisinden 150 satırlık örneklem alınır |

## 5. Ekip için kullanım

Her ekip üyesi kendi model notebook'unun sonunda şu adımları uygular:

1. `explain.py` dosyasını (ve `config.py`, `preprocessing.py`, `metrics.py`) Colab'da sol menüdeki klasör simgesinden yükler.
2. Kendi en iyi modeli ile fonksiyonu çağırır:

```python
%pip install -q shap lime
from explain import explain

result = explain(best_model, X_train, X_test, model_name="SVM")
result["shap_importance"].head(10)   # en önemli 10 özellik
```

3. Grafikleri Bölüm 2'deki okuma rehberine göre yorumlar ve kendi raporuna yazar.

**Aşama 3: tüm modellerin karşılaştırılması.** Herkes modelini bitirdiğinde sonuçlar bir sözlükte toplanır:

```python
from explain import compare_importance

results = {
    "Logistic Regression": explain(lr_model, X_train, X_test, "Logistic Regression", plot=False),
    "KNN": explain(knn_model, X_train, X_test, "KNN", plot=False),
    "SVM": explain(svm_model, X_train, X_test, "SVM", plot=False),
    "Random Forest": explain(rf_model, X_train, X_test, "Random Forest", plot=False),
}
table = compare_importance(results)
table.plot(kind="barh", figsize=(8, 6))
```

Bu tablo, farklı modellerin aynı özelliklere mi yoksa farklı özelliklere mi dayandığını gösterir. Tüm modellerin ortak olarak öne çıkardığı özellikler, inme riski için en güvenilir göstergeler olarak yorumlanabilir.

## 6. Özet (rapor için)

- Kara kutu modellerin kararlarını açıklamak için XAI yöntemleri incelendi: feature importance, permutation importance, SHAP ve LIME.
- Global açıklama için **SHAP**, lokal (tek hasta) açıklama için **LIME** seçildi. SHAP oyun teorisine dayanan sağlam matematiksel temeli ve etkinin yönünü göstermesi, LIME ise modelden bağımsız olması ve tek hasta düzeyinde anlaşılır açıklama vermesi nedeniyle tercih edildi.
- Tüm ekibin kullanabileceği, model türünü otomatik algılayan ortak bir XAI fonksiyonu (`src/explain.py`) yazıldı.
- Her model, eğitildikten sonra bu fonksiyonla açıklanacak ve modellerin özellik önemleri `compare_importance` ile karşılaştırılacaktır.